# Обработка geopqrquete файов для проверки скорости выполнения вычислений на batch

# Инициализация

In [ ]:
import os
import sys

from sedona.spark import SedonaContext

# Явно указываем путь к Python для воркеров
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Настройка Hadoop
os.environ['HADOOP_HOME'] = r'C:\Hadoop\hadoop-3.3.6'

# Пакеты для Spark 3.5.4 со Scala 2.12
additional_packages = [
    "org.apache.sedona:sedona-spark-3.5_2.12:1.8.0",
    "org.datasyslab:geotools-wrapper:1.8.0-33.1"
]

# Оптимальная конфигурация для 24 ядер
config = SedonaContext.builder() \
    .appName("SedonaApp") \
    .master("local[24]") \
    .config("spark.sql.shuffle.partitions", "48") \
    .config("spark.default.parallelism", "48") \
    .config("spark.task.cpus", "1") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryo.registrator", "org.apache.sedona.core.serde.SedonaKryoRegistrator") \
    .config("spark.sql.extensions", "org.apache.sedona.sql.SedonaSqlExtensions,org.apache.sedona.viz.sql.SedonaVizExtensions") \
    .config("spark.jars", r"D:\Artem\Work\amtech_projects\postgresql-42.7.13.jar") \
    .config("spark.jars.packages", ",".join(additional_packages)) \
    .config("spark.driver.memory", "16g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

# Создаем контекст Sedona
sedona = SedonaContext.create(config)


# Декоратор для замера скорости выполнения запроса

In [ ]:
import time
from functools import wraps

def timer_sedona(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"Функция {func.__name__} выполнена за {end - start:.4f} секунд")
        return result
    return wrapper

# Функция для выполнения запросов

In [ ]:
@timer_sedona
def execute_query(session, query):
    return session.sql(query)